# Lab 0-03 Assignment: PII Extraction Against Ground Truth

Three AI assistants will read the same synthetic case file about five fictional people. Each assistant uses a different LLM: `qwen3.5:0.8b`, `qwen3.5:27b`, or `gemma4:e4b`.

## Goal

Test whether the assistants find the same names, phone numbers, email addresses, physical addresses, and device IDs as the known reference answer (the ground truth). Then decide whether the assistants gave the same result and explain why they may differ. All people and records are fictional; do not contact the listed email addresses.

## Step 0: Check Your Setup

Run this block from the `lab0_03_model_basics` folder. It reads the Ollama address from `.env` and prepares the connection for the three assistants.

In [ ]:
import json
from html import escape
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values
from IPython.display import HTML, display
from openai import OpenAI

LAB_NAME = 'lab0_03_model_basics'
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Copy .env.example to .env before continuing.')

config = dotenv_values(env_path)
ollama_base_url = config.get('OLLAMA_BASE_URL')
if not ollama_base_url:
    raise ValueError("OLLAMA_BASE_URL is missing from .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')
print('Ollama address:', ollama_base_url)

## Step 1: Read the Case File and Ground Truth

The text file is the evidence that every assistant will read. `ground_truth` is the instructor's known correct reference answer. It is used after the assistants respond to show what they found, missed, or added.

In [ ]:
case_file_path = lab_dir / 'data' / 'five_people_pii_ground_truth.txt'
case_file = case_file_path.read_text(encoding='utf-8')

# This known answer was created from the synthetic file above.
ground_truth = {
    'names': ['Avery Brooks', 'Naomi Chen', 'Diego Ramirez', 'Priya Shah', 'Marcus Turner'],
    'phone_numbers': ['202-555-0101', '202-555-0102', '202-555-0103', '202-555-0104', '202-555-0105'],
    'email_addresses': ['avery.brooks@google.com', 'naomi.chen@cnn.com', 'diego.ramirez@fbi.gov', 'priya.shah@umd.edu', 'marcus.turner@google.com'],
    'physical_addresses': ['1842 Willow Lane, Madison, WI 53703', '77 Harbor Street, Portland, ME 04101', '910 West Elm Road, Austin, TX 78701', '56 Maple Court, Albany, NY 12207', '300 Pine Avenue, Denver, CO 80202'],
    'device_ids': ['490154203237518', 'TAB-NC-4829', 'DEV-DR-7712', '356938035643809', 'LT-MT-9306'],
}

print(case_file)
print(f'\nGround truth contains {sum(len(values) for values in ground_truth.values())} items.')

## Step 2: Give the Same Task to Three Assistants

Every assistant receives the same case file and instructions. This makes differences in the answers easier to attribute to the models rather than to different questions.

In [ ]:
models_to_compare = ['qwen3.5:0.8b', 'qwen3.5:27b', 'gemma4:e4b']
categories = list(ground_truth)

prompt = f"""
Read the synthetic case file below. Extract names, phone numbers, email addresses, physical addresses, and device IDs.

Return JSON only, using exactly these five keys: {', '.join(categories)}.
Keep each value exactly as it appears in the case file. Use an empty list when a category has no values.

Case file:
{case_file}
""".strip()

def clean_json_text(text):
    text = (text or '').strip().replace('```json', '').replace('```', '').strip()
    start, end = text.find('{'), text.rfind('}')
    return text[start:end + 1] if start >= 0 and end > start else text

def ask_assistant(model_name):
    start = perf_counter()
    response = client.chat.completions.create(model=model_name, messages=[{'role': 'user', 'content': prompt}])
    raw_text = response.choices[0].message.content or ''
    try:
        answer = json.loads(clean_json_text(raw_text))
        if not isinstance(answer, dict):
            raise ValueError('The response was JSON but not a JSON object.')
        error = None
    except (json.JSONDecodeError, ValueError) as exc:
        answer, error = {}, str(exc)
    return {'model': model_name, 'seconds': round(perf_counter() - start, 2), 'answer': answer, 'raw_text': raw_text, 'error': error}

results = []
for model_name in models_to_compare:
    print(f'Asking {model_name}...')
    results.append(ask_assistant(model_name))
print('All three assistants have responded.')

## Step 3: Compare Each Answer with Ground Truth

The table shows how many of the 25 expected items each assistant found exactly, what it missed, and what unexpected items it added. Exact matching makes differences visible, but a formatting difference can also count as a mismatch.

In [ ]:
def normalize(value):
    return ' '.join(str(value).lower().split())

def compare_values(expected, actual):
    expected_map = {normalize(value): value for value in expected}
    actual = actual if isinstance(actual, list) else []
    actual_map = {normalize(value): value for value in actual}
    found = [expected_map[key] for key in expected_map.keys() & actual_map.keys()]
    missed = [expected_map[key] for key in expected_map.keys() - actual_map.keys()]
    extra = [actual_map[key] for key in actual_map.keys() - expected_map.keys()]
    return found, missed, extra

def table_items(items):
    return '<br>'.join(escape(item) for item in items) if items else '—'

expected_total = sum(len(values) for values in ground_truth.values())
analyses = []
for result in results:
    found_all, missed_all, extra_all = [], [], []
    for category in categories:
        found, missed, extra = compare_values(ground_truth[category], result['answer'].get(category, []))
        found_all.extend(found)
        missed_all.extend(f'{category}: {value}' for value in missed)
        extra_all.extend(f'{category}: {value}' for value in extra)
    analyses.append({**result, 'found': found_all, 'missed': missed_all, 'extra': extra_all})

rows = []
for item in analyses:
    json_status = 'Yes' if item['error'] is None else 'No'
    rows.append(f"<tr><td>{escape(item['model'])}</td><td>{item['seconds']}</td><td>{json_status}</td><td>{len(item['found'])}/{expected_total}</td><td>{table_items(item['missed'])}</td><td>{table_items(item['extra'])}</td></tr>")

display(HTML("""<table border='1' cellpadding='6' style='border-collapse:collapse'>
<tr><th>Assistant</th><th>Seconds</th><th>Valid JSON?</th><th>Correct items</th><th>Missed items</th><th>Unexpected items</th></tr>
{}</table>""".format(''.join(rows))))

## Step 4: Do the Assistants Agree?

This check compares the complete structured answers. Read the full answers and the ground-truth table before explaining the result.

In [ ]:
def answer_signature(answer):
    return tuple((category, tuple(sorted(normalize(value) for value in answer.get(category, [])))) for category in categories)

signatures = {item['model']: answer_signature(item['answer']) for item in analyses}
all_same = len(set(signatures.values())) == 1

print('All three assistants produced the same normalized result.' if all_same else 'The assistants did not all produce the same normalized result.')
for item in analyses:
    print(f"{item['model']}: found {len(item['found'])}/{expected_total}, missed {len(item['missed'])}, added {len(item['extra'])}")

## Step 5: Assignment Questions

Answer in a new Markdown cell below:

1. Did all three assistants produce the same results? Use the agreement message and table as evidence.
2. Which assistant found the most ground-truth items? Which was fastest?
3. Give one example of an item that an assistant missed or added unexpectedly.
4. Why might the assistants differ even though they read the same text and instructions? Consider model size, training, instruction following, and formatting choices.
5. What change to the prompt might make the results more consistent?

Save the completed notebook with its output cells and your written answers.